In [1]:
from pathlib import Path

# === PARAMÈTRES ===
root = Path("/media/b085164/Elements/PCD_SAM/longue_ligne/scenario_combined/limatch/merged_2000_HA_LR")
output_file = Path("/media/b085164/Elements/PCD_SAM/longue_ligne/scenario_combined/limatch/merged_2000_HA_LR/LiDAR_p2p.txt")

# === RECHERCHE DES DOSSIERS cor_outputs ===
cor_dirs = sorted(p for p in root.rglob("cor_outputs") if p.is_dir())

# === RECHERCHE DES FICHIERS À MERGER ===
files_to_merge = []
for d in cor_dirs:
    files_to_merge.extend(sorted(d.glob("LiDAR_p2p_chunk*")))

files_to_merge = [f for f in files_to_merge if f.is_file()]

print(f"{len(files_to_merge)} fichiers trouvés.")

# === MERGE ===
output_file.parent.mkdir(parents=True, exist_ok=True)

with output_file.open("w", encoding="utf-8") as fout:
    for f in files_to_merge:
        with f.open("r", encoding="utf-8", errors="replace") as fin:
            content = fin.read()
            fout.write(content)
            if not content.endswith("\n"):
                fout.write("\n")

print(f"Merge terminé : {output_file}")

224 fichiers trouvés.
Merge terminé : /media/b085164/Elements/PCD_SAM/longue_ligne/scenario_combined/limatch/merged_2000_HA_LR/LiDAR_p2p.txt


In [1]:
import numpy as np

file = "/media/b085164/Elements/PCD_SAM/village/merged/merged_800_HA_LR.txt"

data = np.loadtxt(file, delimiter=",")

n_points = data.shape[0]

t = data[:, 0]   # colonne 0 = GPS time
tmin = t.min()
tmax = t.max()

print("Number of points:", n_points)
print("tmin:", tmin)
print("tmax:", tmax)

Number of points: 210570172
tmin: 467364.851373258
tmax: 467573.346314613


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- paths ---
in_path = Path("/media/b085164/Elements/Gobet_ODyN_v1/v4.2_GNSS_outage_limatch_kneighbor1/LiDAR_p2p.txt")      # <-- change to your file
out_path = in_path.with_name(in_path.stem + "_rotvec.txt")

# --- load (CSV with commas) ---
df = pd.read_csv(
    in_path,
    header=None,
    comment="#",
    skip_blank_lines=True
)

# expected columns:
# t_b, t_a, vec_b_x, vec_b_y, vec_b_z, vec_a_x, vec_a_y, vec_a_z
if df.shape[1] < 8:
    raise ValueError(f"Expected at least 8 columns, got {df.shape[1]}")

df = df.iloc[:, :8].copy()
df.columns = ["t_b","t_a","vec_b_x","vec_b_y","vec_b_z","vec_a_x","vec_a_y","vec_a_z"]

R = np.array([[0, 1, 0],
              [1, 0, 0],
              [0, 0,-1]], dtype=float)

# --- apply R to vec_b and vec_a ---
vb = df[["vec_b_x","vec_b_y","vec_b_z"]].to_numpy(float)
va = df[["vec_a_x","vec_a_y","vec_a_z"]].to_numpy(float)

vb_rot = (R @ vb.T).T
va_rot = (R @ va.T).T

df[["vec_b_x","vec_b_y","vec_b_z"]] = vb_rot
df[["vec_a_x","vec_a_y","vec_a_z"]] = va_rot

# --- save ---
df.to_csv(out_path, index=False, header=False, float_format="%.12f")
print("Wrote:", out_path)

# quick sanity check: show first rows
df.head()

Wrote: /media/b085164/Elements/Gobet_ODyN_v1/v4.2_GNSS_outage_limatch_kneighbor1/LiDAR_p2p_rotvec.txt


,t_b,t_a,vec_b_x,vec_b_y,vec_b_z,vec_a_x,vec_a_y,vec_a_z
0,466562.626260,466560.883635,6.6399,-9.8000,-1.4228,7.303,5.250,-1.850
1,466562.636280,466560.758630,7.0620,-10.0634,-1.2329,7.779,5.941,-1.695
2,466562.641253,466560.888630,6.6870,-9.9051,-1.4968,7.368,5.249,-1.920
3,466562.646229,466561.138493,5.7895,-9.0442,-1.5612,6.270,4.001,-1.942
4,466562.651217,466561.213481,5.4587,-8.7352,-1.6041,5.986,3.667,-1.959


## Get times of each recording 

In [5]:
from pathlib import Path
import os

def quick_time_bounds(txt_path, delimiter=","):
    """
    Read GPS time from first and last line of a large TXT file.
    Assumes:
        column 0 = gps time
    """

    txt_path = Path(txt_path)

    with open(txt_path, "rb") as f:

        # ---- FIRST LINE ----
        first_line = f.readline().decode().strip()
        t0 = float(first_line.split(delimiter)[0])

        # ---- LAST LINE (seek from end) ----
        f.seek(-1024, os.SEEK_END)   # jump near end
        tail = f.readlines()

        # find last non-empty line
        for line in reversed(tail):
            if line.strip():
                last_line = line.decode().strip()
                break

        t1 = float(last_line.split(delimiter)[0])

    return t0, t1

In [6]:
merged_dir = Path("/media/b085164/Elements/PCD_SAM/All_pcd/merged")

results = []

for f in sorted(merged_dir.glob("*.txt")):
    try:
        t0, t1 = quick_time_bounds(f)
        results.append((f.name, t0, t1, t1 - t0))
        print(f"{f.name:<40}  [{t0:.3f} → {t1:.3f}]  Δt={t1-t0:.1f}s")
    except Exception as e:
        print(f"{f.name}: FAILED ({e})")

merged_10000_HA_LR.txt                    [467588.721 → 467594.186]  Δt=5.5s
merged_1000_HA_LR.txt                     [466554.821 → 466860.374]  Δt=305.6s
merged_11000_HA_LR.txt                    [467604.016 → 467683.827]  Δt=79.8s
merged_2000_HA_LR.txt                     [466896.472 → 467160.499]  Δt=264.0s
merged_3000_HA_LR.txt                     [467164.153 → 467180.764]  Δt=16.6s
merged_4000_HA_LR.txt                     [467181.482 → 467205.529]  Δt=24.0s
merged_5000_HA_LR.txt                     [467206.034 → 467211.055]  Δt=5.0s
merged_6000_HA_LR.txt                     [467211.507 → 467226.855]  Δt=15.3s
merged_7000_HA_LR.txt                     [467227.556 → 467257.452]  Δt=29.9s
merged_8000_HA_LR.txt                     [467281.138 → 467348.772]  Δt=67.6s
merged_9000_HA_LR.txt                     [467364.851 → 467573.346]  Δt=208.5s


## Quick GNSS outage 

In [7]:
from pathlib import Path

# =====================================================
# INPUT
# =====================================================

gps_in = Path("/media/b085164/Elements/Gobet_ODyN_v1/v1_base_AB/in/GPS.txt")
gps_out = Path("/media/b085164/Elements/Gobet_ODyN_v1/v6/in/GPS.txt")

# outages = (start_time, duration)
outages = [
    (466930.0, 200.0),
]

delimiter = ","


# =====================================================
# BUILD OUTAGE INTERVALS
# =====================================================

intervals = [(s, s + d) for s, d in outages]

print("Removing GPS data during:")
for a, b in intervals:
    print(f"  [{a:.3f} → {b:.3f}]")


# =====================================================
# FILTER GPS FILE (STREAMING → works for huge files)
# =====================================================

kept = 0
removed = 0

with open(gps_in, "r") as fin, open(gps_out, "w") as fout:

    for line in fin:

        if not line.strip():
            continue

        t = float(line.split(delimiter)[0])

        # check if inside outage
        in_outage = any(a <= t <= b for a, b in intervals)

        if in_outage:
            removed += 1
            continue

        fout.write(line)
        kept += 1


print("\nDone.")
print(f"Kept lines   : {kept}")
print(f"Removed lines: {removed}")
print(f"Output file  : {gps_out}")

Removing GPS data during:
  [466930.000 → 467130.000]

Done.
Kept lines   : 3086
Removed lines: 201
Output file  : /media/b085164/Elements/Gobet_ODyN_v1/v6/in/GPS.txt


## Split big PCD spatially

In [2]:
# --- Spatial split into 3 parts (E,N geometry) for TWO huge ASCII point clouds ---
# Assumes each file is a delimited text with at least: [t, E, N, Z, ...]
# You can set which columns are E and N below.

from pathlib import Path
import numpy as np
import pandas as pd
import time

# =========================
# USER CONFIG
# =========================
cloud1 = Path("/media/b085164/Elements/PCD_SAM/longue_ligne/output_Patcher/Flights_1000_2000/Patch_from_scan_1000_with_2000.txt")
cloud2 = Path("/media/b085164/Elements/PCD_SAM/longue_ligne/output_Patcher/Flights_1000_2000/Patch_from_scan_2000_with_1000.txt")

out_dir = Path("/media/b085164/Elements/PCD_SAM/longue_ligne/output_Patcher/output_split")
out_dir.mkdir(parents=True, exist_ok=True)

# delimiter:
# - if whitespace separated: sep = r"\s+" and engine="c"
# - if comma separated: sep = "," and engine="c"
sep = ","          # change to "," if CSV
engine = "c"

# column indices (0-based) for Easting and Northing in your TXT
E_COL = 1
N_COL = 2


CHUNKSIZE = 5_000_000          # try bigger if RAM allows
SAMPLE_TARGET = 600_000        # you can reduce; PCA works well with 200k-1M
SAMPLE_PER_CHUNK = 10_000      # reduce if you want faster first pass

out_sep = "," if sep == "," else " "
float_fmt = "%.6f"

def read_chunks_EN(path: Path):
    for df in pd.read_csv(path, sep=sep, engine=engine, header=None,
                          usecols=[E_COL, N_COL], dtype=np.float64,
                          chunksize=CHUNKSIZE, low_memory=False):
        yield df

def estimate_axis_and_thresholds_from_sample(paths):
    rng = np.random.default_rng(0)
    samples = []
    n_total = 0
    t0 = time.time()

    for p in paths:
        for df in read_chunks_EN(p):
            arr = df.to_numpy(copy=False)
            m = arr.shape[0]
            if m == 0:
                continue
            take = min(SAMPLE_PER_CHUNK, m)
            idx = rng.choice(m, size=take, replace=False)
            samples.append(arr[idx])
            n_total += take
            if n_total >= SAMPLE_TARGET:
                break
        if n_total >= SAMPLE_TARGET:
            break

    X = np.vstack(samples)
    mu = X.mean(axis=0)
    Xc = X - mu
    C = (Xc.T @ Xc) / max(1, (Xc.shape[0] - 1))
    evals, evecs = np.linalg.eigh(C)
    u = evecs[:, np.argmax(evals)]
    u /= np.linalg.norm(u)

    proj = X @ u
    # quantiles on sample => 3 roughly equal parts
    t1, t2 = np.quantile(proj, [1/3, 2/3])

    print(f"[FAST PCA] sample={X.shape[0]:,}, u=[{u[0]:.6f},{u[1]:.6f}], t1={t1:.3f}, t2={t2:.3f} (t={time.time()-t0:.1f}s)")
    return u, float(t1), float(t2)

def split_file_into_3(path_in: Path, u, t1, t2, out_paths):
    outs = [open(p, "w", buffering=1024*1024) for p in out_paths]
    try:
        t0 = time.time()
        n_written = [0,0,0]
        n_total = 0

        for df in pd.read_csv(path_in, sep=sep, engine=engine, header=None,
                              dtype=np.float64, chunksize=CHUNKSIZE, low_memory=False):
            arr = df.to_numpy(copy=False)
            if arr.size == 0:
                continue

            EN = arr[:, [E_COL, N_COL]]
            proj = EN @ u

            m0 = proj < t1
            m1 = (proj >= t1) & (proj < t2)
            m2 = proj >= t2

            if m0.any():
                np.savetxt(outs[0], arr[m0], delimiter=out_sep, fmt=float_fmt); n_written[0]+=int(m0.sum())
            if m1.any():
                np.savetxt(outs[1], arr[m1], delimiter=out_sep, fmt=float_fmt); n_written[1]+=int(m1.sum())
            if m2.any():
                np.savetxt(outs[2], arr[m2], delimiter=out_sep, fmt=float_fmt); n_written[2]+=int(m2.sum())

            n_total += arr.shape[0]
            if n_total % (5*CHUNKSIZE) == 0:
                print(f"  {path_in.name}: processed {n_total:,} rows...")

        print(f"[SPLIT] {path_in.name}: {n_written[0]:,}/{n_written[1]:,}/{n_written[2]:,} (t={time.time()-t0:.1f}s)")
    finally:
        for f in outs: f.close()

u, t1, t2 = estimate_axis_and_thresholds_from_sample([cloud1, cloud2])

c1_out = [out_dir/f"{cloud1.stem}__part1.txt", out_dir/f"{cloud1.stem}__part2.txt", out_dir/f"{cloud1.stem}__part3.txt"]
c2_out = [out_dir/f"{cloud2.stem}__part1.txt", out_dir/f"{cloud2.stem}__part2.txt", out_dir/f"{cloud2.stem}__part3.txt"]

print("\n--- Splitting cloud 1 ---")
split_file_into_3(cloud1, u, t1, t2, c1_out)

print("\n--- Splitting cloud 2 ---")
split_file_into_3(cloud2, u, t1, t2, c2_out)

print("\nDone.")

[FAST PCA] sample=600,000, u=[-0.948999,-0.315278], t1=-2780962.986, t2=-2779498.092 (t=89.3s)

--- Splitting cloud 1 ---
  Patch_from_scan_1000_with_2000.txt: processed 25,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 50,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 75,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 100,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 125,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 150,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 175,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 200,000,000 rows...
  Patch_from_scan_1000_with_2000.txt: processed 225,000,000 rows...
[SPLIT] Patch_from_scan_1000_with_2000.txt: 43,375,613/84,282,195/100,012,993 (t=546.2s)

--- Splitting cloud 2 ---
  Patch_from_scan_2000_with_1000.txt: processed 25,000,000 rows...
  Patch_from_scan_2000_with_1000.txt: processed 50,000,000 rows...
  Patch_from_scan_2

In [3]:
from pathlib import Path

# adapte cet import selon où tu exécutes le notebook
from singleBeamMerging import merge_txt_pairs

base_root = Path("/media/b085164/Elements/PCD_SAM/Georef_v6")
scenarios = ["ref", "outage_only"]

for scenario in scenarios:
    scenario_dir = base_root / scenario
    dir_ha = scenario_dir / "HA"
    dir_lr = scenario_dir / "LR"
    out_dir = scenario_dir / "merged"

    print("\n" + "=" * 70)
    print(f"[MERGING SCENARIO] {scenario}")
    print(f"HA dir   : {dir_ha}")
    print(f"LR dir   : {dir_lr}")
    print(f"OUT dir  : {out_dir}")
    print("=" * 70)

    merge_txt_pairs(
        dir_a=dir_ha,
        dir_b=dir_lr,
        out_dir=out_dir,
        delimiter=",",
        skiprows=0,
        sort_by_time=True,
        out_prefix="merged_",
        out_suffix="_HA_LR",
    )

print("\n[DONE] ref + outage_only merged.")


[MERGING SCENARIO] ref
HA dir   : /media/b085164/Elements/PCD_SAM/Georef_v6/ref/HA
LR dir   : /media/b085164/Elements/PCD_SAM/Georef_v6/ref/LR
OUT dir  : /media/b085164/Elements/PCD_SAM/Georef_v6/ref/merged

 [Merging] files /media/b085164/Elements/PCD_SAM/Georef_v6/ref/HA/250220_094117_VUX-HA1_pcd.txt and /media/b085164/Elements/PCD_SAM/Georef_v6/ref/LR/250220_094117_VUX1-LR_pcd.txt
250220_094117_VUX-HA1_pcd.txt + 250220_094117_VUX1-LR_pcd.txt -> merged_94117_HA_LR.txt (140986168 + 120925420 rows)

--- Summary ---
[Merging] Pairs found: 1 | Merged: 1
[Merging] Output dir: /media/b085164/Elements/PCD_SAM/Georef_v6/ref/merged

[MERGING SCENARIO] outage_only
HA dir   : /media/b085164/Elements/PCD_SAM/Georef_v6/outage_only/HA
LR dir   : /media/b085164/Elements/PCD_SAM/Georef_v6/outage_only/LR
OUT dir  : /media/b085164/Elements/PCD_SAM/Georef_v6/outage_only/merged

 [Merging] files /media/b085164/Elements/PCD_SAM/Georef_v6/outage_only/HA/250220_094117_VUX-HA1_pcd.txt and /media/b085164/El

In [ ]:
from pathlib import Path
from tqdm import tqdm

# -------------------------------------------------------------------
# ROOT FOLDER TO SCAN
# -------------------------------------------------------------------
root_folder = Path("/media/b085164/Elements/PCD_SAM/Georef_v6/")

# -------------------------------------------------------------------
# FUNCTION: fast line counting (streaming)
# -------------------------------------------------------------------
def count_lines_fast(file_path, chunk_size=1024*1024*16):
    count = 0
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            count += chunk.count(b"\n")
    return count

# -------------------------------------------------------------------
# FIND ALL TXT FILES
# -------------------------------------------------------------------
txt_files = sorted(root_folder.rglob("*.txt"))

print(f"Found {len(txt_files)} txt files\n")

results = []

for f in tqdm(txt_files, desc="Counting lines"):
    try:
        n = count_lines_fast(f)
        results.append((str(f), n))
    except Exception as e:
        results.append((str(f), f"ERROR: {e}"))

# -------------------------------------------------------------------
# PRINT RESULTS
# -------------------------------------------------------------------
print("\n================ LINE COUNTS ================\n")

for path, n in results:
    print(f"{n:>15}  |  {path}")


Found 4 txt files



Counting lines: 100%|██████████| 4/4 [16:17<00:00, 244.38s/it]



================ LINE COUNTS ================

Found 4 txt files



Counting lines: 100%|██████████| 4/4 [19:00<00:00, 285.16s/it]


================ LINE COUNTS ================

      261911575  |  /media/b085164/Elements/PCD_SAM/Georef_v6/limatch_chunk2chunk/merged/merged_94117_HA_LR.txt
      261911762  |  /media/b085164/Elements/PCD_SAM/Georef_v6/limatch_scan2scan/merged/merged_94117_HA_LR.txt
      261911555  |  /media/b085164/Elements/PCD_SAM/Georef_v6/outage_only/merged/merged_94117_HA_LR.txt
      261911588  |  /media/b085164/Elements/PCD_SAM/Georef_v6/ref/merged/merged_94117_HA_LR.txt


In [ ]:
from pathlib import Path

def first_last_line(path):
    path = Path(path)

    with open(path, "rb") as f:
        first = f.readline().decode("utf-8", errors="ignore").strip()

        f.seek(0, 2)
        pos = f.tell() - 1
        while pos > 0:
            f.seek(pos)
            if f.read(1) == b"\n":
                break
            pos -= 1
        last = f.readline().decode("utf-8", errors="ignore").strip()

    return first, last

def parse_time_from_txt_line(line, delim=",", time_col=0):
    parts = [x.strip() for x in line.split(delim)]
    return float(parts[time_col])

ref_txt = "/media/b085164/Elements/PCD_SAM/Georef_v6/ref/merged/merged_94117_HA_LR.txt"
tgt_txt = "/media/b085164/Elements/PCD_SAM/Georef_v6/outage_only/merged/merged_94117_HA_LR.txt"

for label, path in [("REF", ref_txt), ("TGT", tgt_txt)]:
    first, last = first_last_line(path)
    t0 = parse_time_from_txt_line(first, delim=",", time_col=0)
    t1 = parse_time_from_txt_line(last,  delim=",", time_col=0)
    print(f"{label}")
    print(f"  first line: {first[:120]}")
    print(f"  last  line: {last[:120]}")
    print(f"  time range: {t0:.6f} -> {t1:.6f}")
    print()

FileNotFoundError: [Errno 2] No such file or directory: '/media/b085164/Elements/PCD_SAM/Georef_v6/reference/merged/merged_94117_HA_LR.txt'